# Censoring sensitivity analysis

In [ ]:
import pandas as pd

In [ ]:
def get_files_fold(data_src, fold):
    clinical_data_source = f'data_files/{data_src}/splits/{fold}/train_filtered.csv'
    out_dir = f'data_files/{data_src}/splits/{fold}'
    data_df = pd.read_csv(clinical_data_source) 
    total = len(data_df)
    n_observed = (data_df["dss_censorship"] == 0.0).sum()
    n_censored = (data_df["dss_censorship"] == 1.0).sum()

    print(f"Total samples: {total}, Observed events: {n_observed}, Censored events: {n_censored}")
    print(f"Current censoring rate: {n_censored/total:.2f}\n")

    results = []

    for censor_rate in [0.5, 0.6, 0.7, 0.8, 0.9]:
        
        # Undersampling
        n_censored_target = int(n_observed * censor_rate / (1 - censor_rate))
        n_to_remove = n_censored - n_censored_target
        
        if n_to_remove >= 0:
            censored_to_remove = data_df[data_df["dss_censorship"] == 1.0].sample(n=n_to_remove, random_state=42)
            df_under = data_df.drop(censored_to_remove.index).reset_index(drop=True)
            actual_rate_under = (df_under["dss_censorship"] == 1.0).sum() / len(df_under)
            print(f"[Undersampling] Target: {censor_rate}, Actual: {actual_rate_under:.2f}, Total samples: {len(df_under)}")
        else:
            df_under = None
            print(f"[Undersampling] Target: {censor_rate}: not achievable")

        # Oversampling
        n_observed_target = int(n_censored * (1 - censor_rate) / censor_rate)
        n_to_duplicate = n_observed_target - n_observed
        
        if n_to_duplicate >= 0:
            duplicated = data_df[data_df["dss_censorship"] == 0.0].sample(n=n_to_duplicate, replace=True, random_state=42)
            df_over = pd.concat([data_df, duplicated]).reset_index(drop=True)
            actual_rate_over = (df_over["dss_censorship"] == 1.0).sum() / len(df_over)
            print(f"[Oversampling]  Target: {censor_rate}, Actual: {actual_rate_over:.2f}, Total samples: {len(df_over)}")
        else:
            df_over = None
            print(f"[Oversampling]  Target: {censor_rate}: not achievable")

        results.append({
            "target_censor_rate": censor_rate,
            "df_undersampled": df_under,
            "df_oversampled": df_over
        })

        print()
    

    for res in results:
        censor_rate = res["target_censor_rate"]
        
        if res["df_undersampled"] is not None:
            out_file_under = f'{out_dir}/train_filtered_undersampled_censor_{int(censor_rate*100)}.csv'
            res["df_undersampled"].to_csv(out_file_under, index=False)
        
        if res["df_oversampled"] is not None:
            out_file_over = f'{out_dir}/train_filtered_oversampled_censor_{int(censor_rate*100)}.csv'
            res["df_oversampled"].to_csv(out_file_over, index=False)


In [ ]:
# Check nr observed events test sets
for data_src in ['tcga_brca', 'tcga_blca', 'tcga_luad', 'tcga_kirc']:
    print(f"Data source: {data_src}")
    for i in range(5):
        clinical_data_source = f'data_files/{data_src}/splits/{i}/train_filtered.csv'
        data_df = pd.read_csv(clinical_data_source) 
        total = len(data_df)
        n_observed = (data_df["dss_censorship"] == 0.0).sum()
        
        print(f"Fold {i} Train - Total samples: {total}, Observed events: {n_observed}, Censoring rate: {(total-n_observed)/total:.2f}")

        clinical_data_source = f'data_files/{data_src}/splits/{i}/test_filtered.csv'
        data_df = pd.read_csv(clinical_data_source) 
        total = len(data_df)
        n_observed = (data_df["dss_censorship"] == 0.0).sum()
        print(f"Fold {i} Test - Total samples: {total}, Observed events: {n_observed}, Censoring rate: {(total-n_observed)/total:.2f}")
